# Cell Tracking Analysis
This notebook demonstrates the cell tracking analysis process using our custom modules.

In [ ]:
%pip install opencv-python
%pip install numpy
%pip install tifffile
%pip install matplotlib
%pip install seaborn
%pip install pandas
%pip install Pillow
%pip install tqdm
%pip install ipywidgets
%pip install jupyter

In [ ]:
import sys
from pathlib import Path
import time
from datetime import datetime
import os

# Add Source directory to Python path
source_dir = Path('Source')
sys.path.append(str(source_dir.absolute()))

# Import our custom modules
from blob_tracker import BlobTracker
from metrics_calculator import CellTrackingMetrics
from image_processor import ImageProcessor
from utils import setup_logging, get_input_file

## Setup Output Directories

In [ ]:
# Setup paths
data_dir = Path('.')
output_folder = data_dir / 'Outputs/Final'
log_folder = output_folder / 'logs'

# Create output directories
output_folder.mkdir(exist_ok=True, parents=True)
log_folder.mkdir(exist_ok=True)

# Setup logging
logger = setup_logging(log_folder)

## Get Input File

In [ ]:
print(f"Current working directory: {os.getcwd()}")

# Get input file
input_path = get_input_file()
print(f"Selected input file: {input_path}")

## Initialize Processing Classes

In [ ]:
# Create instances of our classes
tracker = BlobTracker()
metrics_calculator = CellTrackingMetrics()
image_processor = ImageProcessor(logger)

# Create output paths
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
output_name = input_path.stem
output_dir = output_folder / output_name
output_dir.mkdir(exist_ok=True)
output_path = output_dir / f"tracked_cells_{timestamp}.tiff"
plots_dir = output_dir / 'movement_plots'
plots_dir.mkdir(exist_ok=True)

## Process TIFF Stack

In [ ]:
start_time = time.time()

logger.info("Starting cell tracking analysis")
logger.info(f"Input file: {input_path}")
logger.info(f"Output directory: {output_dir}")

# Process TIFF stack
num_frames = image_processor.process_tiff_stack(input_path, output_path, tracker, metrics_calculator)

## Calculate and Display Metrics

In [ ]:
# Calculate and display metrics for Cell ID 2
logger.info("Calculating and plotting metrics for Cell ID 2...")

if 2 in tracker.get_tracks():
    positions = tracker.get_tracks()[2]
    metrics = metrics_calculator.calculate_metrics(positions, num_frames)

    if metrics:
        print("\n=== Metrics for Cell ID 2 ===")
        print(f"Total Distance: {metrics['total_distance']:.2f} pixels")
        print(f"Net Displacement: {metrics['displacement']:.2f} pixels")
        print(f"Average Velocity: {metrics['avg_velocity']:.2f} pixels/frame")
        print(f"Maximum Velocity: {metrics['max_velocity']:.2f} pixels/frame")
        print(f"Average Acceleration: {metrics['avg_acceleration']:.2f} pixels/frame²")
        print(f"Direction Changes: {metrics['directional_changes']}")
        print(f"Confinement Ratio: {metrics['confinement_ratio']:.3f}")

        # Generate and display plots
        metrics_calculator.plot_movement_graphs(2, metrics, plots_dir)
else:
    logger.warning("Cell ID 2 not found in the tracked cells.")

end_time = time.time()
logger.info(f"Processing completed in {end_time - start_time:.2f} seconds.")